In [7]:
# =====================================================================================
# FYP_DataAnalysis
# The Impact of Corporate ESG Disclosure on Corporate Investment Efficiency
# Author: Jiayu Hu
#
# ENVIRONMENT
#   pip install pandas "numpy<2" statsmodels linearmodels pydynpd openpyxl scipy
#   WARNING: pydynpd (system GMM) is currently incompatible with numpy>=2; use numpy 1.26.x.
#
# The file can be run section by section as `# %%` cells in VS Code / Jupyter, or all at once.
# =====================================================================================

# %% [0] Environment and dependencies -------------------------------------------------
%pip install pandas "numpy<2" statsmodels linearmodels pydynpd openpyxl scipy
import sys
!{sys.executable} -m pip install "xlrd>=2.0.1"

import os
import warnings
import numpy as np
import pandas as pd

# pydynpd still calls np.in1d, which was removed in numpy 2.0; keep a compatibility
# shim (harmless under numpy<2).
if not hasattr(np, "in1d"):
    np.in1d = np.isin  # noqa

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from linearmodels.iv import IV2SLS

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

WORKDIR = os.environ.get("FYP_DIR", ".")
os.chdir(WORKDIR)


# %% [0.1] Generic helper functions ---------------------------------------------------
def winsorize(s: pd.Series, lower_pct: float = 0.01, upper_pct: float = 0.99) -> pd.Series:
    """1%/99% winsorization. Equivalent to the R custom_winsorize (type-7 quantiles,
    which matches the pandas default)."""
    lo, hi = s.quantile(lower_pct), s.quantile(upper_pct)
    return s.clip(lower=lo, upper=hi)


def pad_symbol(s: pd.Series) -> pd.Series:
    """Zero-pad the stock code to a 6-character string. Equivalent to R's
    sprintf('%06s', Symbol)."""
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(6)


# List of control variables (consistent with the report)
CONTROLS = ["Size", "PB", "Lev", "ROA", "Zsorcing", "LA", "Board", "IDR",
            "Age", "Female", "FB", "OB", "TOP", "SECOND", "DUAL", "SOE"]


def fe_ols(df, y, x_vars, controls=None, fe=("Industry", "Year"),
           cluster="Symbol", add_const=True):
    """
    OLS with fixed effects — reproduces the "Year FE: Yes / Industry FE: Yes" rows in
    the report's tables. Uses C(Industry)+C(Year) to absorb the fixed effects; standard
    errors are clustered by firm (Symbol) by default.
    (The original R lm() had no fixed effects, so its coefficients / significance differ
     from the report; the report used Stata FE estimators such as reghdfe/areg, which is
     reproduced here with statsmodels.)

    Returns a statsmodels regression-results object.
    """
    controls = controls or []
    sub = df.dropna(subset=[y] + list(x_vars) + list(controls) + list(fe)).copy()
    # Cast the FE columns to category and pass them directly into the formula
    # (avoids depending on patsy's C() being shadowed in the namespace).
    for f in fe:
        sub[f] = sub[f].astype("category")
    rhs = list(x_vars) + list(controls) + list(fe)
    formula = f"{y} ~ " + " + ".join(rhs)
    if not add_const:
        formula += " - 1"
    if cluster and cluster in sub.columns:
        res = smf.ols(formula, data=sub).fit(
            cov_type="cluster", cov_kwds={"groups": sub[cluster]})
    else:
        res = smf.ols(formula, data=sub).fit(cov_type="HC1")
    return res


def tidy(res, keep=("ESGR", "TRANS", "ESGID", "ASY2"), label=""):
    """Print only the core coefficients we care about (hides the many FE dummies),
    making it easy to compare against the report."""
    rows = []
    for v in res.params.index:
        base = v.split("[")[0]
        if base in keep or base in CONTROLS or base == "Intercept":
            rows.append((v, res.params[v], res.tvalues[v], res.pvalues[v]))
    out = pd.DataFrame(rows, columns=["var", "coef", "t", "p"]).set_index("var")
    out["sig"] = pd.cut(out["p"], [-1, .01, .05, .1, 1], labels=["***", "**", "*", ""])
    if label:
        print(f"\n=== {label} | N={int(res.nobs)} | adj_R2={res.rsquared_adj:.4f} ===")
    return out.round(4)


# %% [1] Read the data ----------------------------------------------------------------
# NOTE: the Chinese file names below are the actual files on disk and are kept as-is.
Rechardson = pd.read_excel("Rechardson.xlsx")
Biddle     = pd.read_excel("Biddle.xlsx")
Chen       = pd.read_excel("Chen.xlsx")
ESG_IV     = pd.read_excel("华证ESG及工具变量.xlsx")   # Sino-Securities ESG & instruments
CV         = pd.read_excel("Control Variables.xlsx")
ZS         = pd.read_excel("Zsorcing.xlsx")
IA         = pd.read_excel("ASY.xlsx")
ESG34      = pd.read_excel("ESG分歧34.xls")            # ESG rating divergence


# %% [2] Cleaning and type conversion -------------------------------------------------
Rechardson = Rechardson.drop_duplicates(subset=["Symbol", "Year"], keep="first").copy()

# Year may be either a date (Rechardson/ZS) or a number; extract the year as an integer.
def year_to_int(s):
    dt = pd.to_datetime(s, errors="coerce")
    return dt.dt.year.fillna(pd.to_numeric(s, errors="coerce")).astype("Int64").astype(int)

Rechardson["Year"] = year_to_int(Rechardson["Year"])
Rechardson["Symbol"] = pad_symbol(Rechardson["Symbol"])

for d in (Biddle, Chen, CV, ESG34):
    d["Symbol"] = pad_symbol(d["Symbol"])
    d["Year"] = pd.to_numeric(d["Year"], errors="coerce").astype("Int64").astype(int)

ZS["Year"] = year_to_int(ZS["Year"])
ZS["Symbol"] = pad_symbol(ZS["Symbol"])

for d in (IA, ESG_IV):
    if "Symbol" in d.columns:
        d["Symbol"] = pad_symbol(d["Symbol"])
    d["Year"] = pd.to_numeric(d["Year"], errors="coerce").astype("Int64").astype(int)


# %% [3] Year filter 2013-2023 --------------------------------------------------------
def yrfilter(d):
    return d[(d["Year"] >= 2013) & (d["Year"] <= 2023)].copy()

Rechardson1, Biddle1, Chen1 = map(yrfilter, (Rechardson, Biddle, Chen))
CV1, ZS1, IA1 = map(yrfilter, (CV, ZS, IA))
ESG_IV1, ESG341 = map(yrfilter, (ESG_IV, ESG34))


# %% [4] Select and rename variables --------------------------------------------------
# IE1: Richardson (2006) investment efficiency; also keep the over/under-investment flag.
Rechardson2 = (Rechardson1[["Symbol", "Year", "IndustryName",
                            "InefficInvestDegree", "InefficInvestSign"]]
               .rename(columns={"InefficInvestDegree": "IE1",
                                "InefficInvestSign": "Over_or_Under_Investment"}))

Biddle2 = Biddle1[["Symbol", "Year", "Inveffi"]].rename(columns={"Inveffi": "IE2"})
Chen2   = Chen1[["Symbol", "Year", "Inveffi"]].rename(columns={"Inveffi": "IE3"})

CV2 = (CV1[["Symbol", "Year", "Size", "PB", "Lev", "ROA1", "ListAge", "Boardsize",
            "IndDirectorRatio", "AverageAge", "MaleRatio", "MngmFinancialBack",
            "MngmOverseaBack", "Shrcr1", "Shrz", "Dual", "ContrshrNature", "TobinQ"]]
       .rename(columns={"ContrshrNature": "SOE", "ROA1": "ROA", "ListAge": "LA",
                        "Boardsize": "Board", "IndDirectorRatio": "IDR",
                        "AverageAge": "Age", "MngmFinancialBack": "FB",
                        "MngmOverseaBack": "OB", "Shrcr1": "TOP", "Dual": "DUAL"}))

ZS2 = ZS1[["Symbol", "Year", "Zsorcing"]].copy()
IA2 = IA1[["Symbol", "Year", "LR", "ILL", "GAM"]].copy()   # PCA is computed later in [9]
# Instrument source: mean1..mean3 = industry-year / province-year / industry-province-year
# average ESG (ESG_1 / ESG_2 / ESG_3). "ESG评级赋值" is the assigned ESG rating column.
ESG_IV2 = (ESG_IV1[["Symbol", "Year", "ESG评级赋值",
                    "mean1", "mean2", "mean3", "mean4", "mean5",
                    "mean6", "mean7", "mean8", "mean9", "mean10", "IndustryCode"]]
           .rename(columns={"IndustryCode": "Industry"}))

ESG342 = ESG341[["Symbol", "Year", "ESGdif6", "ESGmin1", "ESGmax1", "ESGrange6"]].copy()


# %% [5] Impute missing ZS values with the firm mean ----------------------------------
ZS2["Zsorcing"] = ZS2.groupby("Symbol")["Zsorcing"].transform(
    lambda x: x.fillna(x.mean()))
ZS2 = ZS2.groupby("Symbol").filter(lambda g: g["Zsorcing"].notna().any())


# %% [6] Merge all datasets -----------------------------------------------------------
def lj(left, right):
    m = left.merge(right, on=["Symbol", "Year"], how="left")
    return m.drop_duplicates(subset=["Symbol", "Year"], keep="first")

merged = Rechardson2
for r in (CV2, Biddle2, Chen2, ZS2, IA2, ESG_IV2, ESG342):
    merged = lj(merged, r)


# %% [7] Drop missing, negate IE, build Female / SECOND -------------------------------
Statement = merged.dropna().copy()

# Investment efficiency = negative of the absolute residual -> larger value = more efficient
Statement[["IE1", "IE2", "IE3"]] = -Statement[["IE1", "IE2", "IE3"]]

# Female = 100 - male ratio; SECOND = (1/Shrz)*TOP (second-largest shareholder's stake)
Statement["Female"] = 100 - Statement["MaleRatio"]
Statement["SECOND"] = (1 / Statement["Shrz"]) * Statement["TOP"]


# %% [8] 1%/99% winsorization ---------------------------------------------------------.
# NOTE: "ESG评级赋值" is the original ESG-rating column (renamed to ESGR below).
wins_cols = (["IE1", "IE2", "IE3", "TobinQ", "ESG评级赋值",
              "ESGdif6", "ESGmin1", "ESGmax1", "ESGrange6"]
             + ["Size", "PB", "Lev", "ROA", "Zsorcing", "LA", "Board", "IDR",
                "Age", "Female", "MaleRatio", "TOP", "SECOND"]
             + [f"mean{i}" for i in range(1, 11)])
for c in wins_cols:
    if c in Statement.columns:
        Statement[c] = winsorize(Statement[c])

# IV alias: the report calls it ESGR; the original data column is "ESG评级赋值".
Statement["ESGR"] = Statement["ESG评级赋值"]


# %% [9] ===== [STATA->PY ADDED] Information-asymmetry PCA: LR/ILL/GAM -> ASY1/ASY2 ====
# Report 3.2.3 + Table 1a: run PCA on LR, ILL, GAM; take PC1=ASY1, PC2=ASY2
def run_asym_pca(df, cols=("LR", "ILL", "GAM")):
    X = df[list(cols)].astype(float)
    Z = (X - X.mean()) / X.std(ddof=0)              # standardize (matches Stata pca on the corr. matrix)
    C = np.corrcoef(Z.values, rowvar=False)
    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    loadings = pd.DataFrame(vecs, index=list(cols),
                            columns=["PC1", "PC2", "PC3"])

    if loadings.loc["LR", "PC2"] > 0:
        loadings["PC2"] *= -1
    if loadings.loc["ILL", "PC1"] < 0:
        loadings["PC1"] *= -1
    scores = Z.values @ loadings.values
    expl = vals / vals.sum()
    cum = np.cumsum(expl)

    var_tbl = pd.DataFrame({
        "Eigenvalue":       vals,
        "Prop. variance %": expl * 100,
        "Cumulative %":     cum * 100,
    }, index=["PC1", "PC2", "PC3"]).round(2)
    print("Information-asymmetry PCA -- variance explained "
          "(report Table 1a: PC1=46.08%, PC1+PC2=78.87%):")
    print(var_tbl)
    print(f"  -> reproduced: PC1={cum[0]*100:.2f}%, PC1+PC2={cum[1]*100:.2f}%")
    print("\nComponent loadings (compare with Table 1a):")
    print(loadings.round(4))
    return scores, loadings

_scores, _loadings = run_asym_pca(Statement)
Statement["ASY1"] = _scores[:, 0]      # first principal component
Statement["ASY2"] = _scores[:, 1]      # second principal component
Statement["TRANS"] = -Statement["ASY1"]   # transparency = negative of information asymmetry (report & R agree)


# %% [10] Descriptive statistics -- Table 2 -------------------------------------------
def describe_table(df, cols):
    d = df[cols].agg(["mean", "std", "min",
                      lambda s: s.quantile(.25),
                      "median",
                      lambda s: s.quantile(.75),
                      "max"]).T
    d.columns = ["Mean", "SD", "Min", "1stQu", "Median", "3rdQu", "Max"]
    return d.round(2)

desc_vars = ["IE1", "IE2", "IE3", "ESGR", "TRANS"] + CONTROLS
print(f"\nN = {len(Statement)}")
print(describe_table(Statement, desc_vars))


# %% [11] Correlation matrix -- Table 3 -----------------------------------------------
corr_vars = ["IE1", "IE2", "IE3", "ESGR"] + CONTROLS
corr_matrix = Statement[corr_vars].corr().round(2)
print("\nPearson correlation matrix (Table 3):")
print(corr_matrix)


# %% [12] Variance inflation factor (VIF) -- Table 4 ----------------------------------
def vif_table(df, xvars):
    X = sm.add_constant(df[xvars].astype(float).dropna())
    out = pd.DataFrame({
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }, index=X.columns)
    return out.drop("const").round(2)

print("\nVIF (Table 4):")
print(vif_table(Statement, ["ESGR"] + CONTROLS))


# %% [13] H1: ESG disclosure -> investment efficiency (with Year FE + Industry FE) -- Tables 5-7
# This is the FE version.
for ie, tbl in [("IE1", "Table 5"), ("IE2", "Table 6"), ("IE3", "Table 7")]:
    m_no  = fe_ols(Statement, ie, ["ESGR"], controls=None)          # Model 1: no controls
    m_yes = fe_ols(Statement, ie, ["ESGR"], controls=CONTROLS)      # Model 2: with controls
    print(tidy(m_no,  label=f"{tbl} {ie} | Model 1 (no controls)").loc[["Intercept", "ESGR"]])
    print(tidy(m_yes, label=f"{tbl} {ie} | Model 2 (with controls)"))


# %% [14] H2: mediating effect of transparency (TRANS) (Baron & Kenny, 1986) -- Table 8
# Step1: TRANS ~ ESGR + controls + FE
# Step2: IE    ~ ESGR + TRANS + controls + FE
# We use TRANS (= -ASY1) so the signs match the positive coefficients in report Table 8
m_trans = fe_ols(Statement, "TRANS", ["ESGR"], controls=CONTROLS)
# Table 8 reports ALL control-variable coefficients (no .loc row slicing),
# matching the full layout of Table 8 in the report.
print(tidy(m_trans, label="Table 8 Model 1: TRANS ~ ESGR"))

for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    m = fe_ols(Statement, ie, ["ESGR", "TRANS"], controls=CONTROLS)
    print(tidy(m, label=f"Table 8 {mlbl}: {ie} ~ ESGR + TRANS"))


# %% [15] Export the cleaned sample (original R: write_xlsx(Statement, "Statement.xlsx"))
Statement.to_excel("Statement.xlsx", index=False)
print("\nExported Statement.xlsx")


# %% [16] ===== [STATA->PY ADDED] Robustness: replace IV with ESGID (first PC) -- Table 9
# TODO: set AGENCY_COLS to the 6 agency-rating column names in ESG分歧34.xls.
AGENCY_COLS = ["华证", "Wind", "富时罗素", "盟浪", "商道融绿", "msci"]

if all(c in ESG341.columns for c in AGENCY_COLS):
    # pull the 6 agency ratings (Symbol already zero-padded in step [2]) and merge
    _ag = ESG341[["Symbol", "Year"] + AGENCY_COLS].drop_duplicates(["Symbol", "Year"])
    St9 = Statement.merge(_ag, on=["Symbol", "Year"], how="left").dropna(subset=AGENCY_COLS)

    # standardize -> PCA -> first principal component
    Xg = St9[AGENCY_COLS].astype(float)
    Zg = (Xg - Xg.mean()) / Xg.std(ddof=0)
    Cg = np.corrcoef(Zg.values, rowvar=False)
    valg, vecg = np.linalg.eigh(Cg)
    order = np.argsort(valg)[::-1]
    pc1_load = vecg[:, order[0]]
    esgid = Zg.values @ pc1_load
    # sign alignment: make ESGID move WITH the original ESGR (higher = better ESG),
    # so the H1/H2 coefficients are directionally interpretable (PCA sign is arbitrary).
    if np.corrcoef(esgid, St9["ESGR"])[0, 1] < 0:
        pc1_load, esgid = -pc1_load, -esgid
    St9["ESGID"] = winsorize(pd.Series(esgid, index=St9.index))

    # variance + loadings check (how much the 1st PC captures, and each agency's weight)
    expl = valg[order] / valg.sum()
    print("ESGID PCA (6 agencies) -- variance explained:")
    print(pd.DataFrame({"Prop.%": expl * 100, "Cum.%": np.cumsum(expl) * 100},
                       index=[f"PC{i+1}" for i in range(len(AGENCY_COLS))]).round(2))
    print("PC1 loadings:", dict(zip(AGENCY_COLS, np.round(pc1_load, 3))))

    print(tidy(fe_ols(St9, "TRANS", ["ESGID"], CONTROLS),
               label="Table 9 Model1: TRANS ~ ESGID").loc[["Intercept", "ESGID"]])
    for ie, mlbl in [("IE1", "Model2"), ("IE2", "Model3"), ("IE3", "Model4")]:
        m = fe_ols(St9, ie, ["ESGID", "TRANS"], CONTROLS)
        print(tidy(m, label=f"Table 9 {mlbl}: {ie} ~ ESGID + TRANS").loc[["Intercept", "ESGID", "TRANS"]])
else:
    print("\n[Skipped Table 9] Set AGENCY_COLS to the 6 agency-rating columns in ESG分歧34.xls.")
    print("  Available columns in ESG分歧34.xls:", list(ESG341.columns))


# %% [17] Endogeneity tests -- Table 10 -----------------------------------------------
# Instruments: ESG_1=industry-year mean ESG (mean1), ESG_2=province-year mean ESG (mean2),
#              ESG_3=industry-province-year mean ESG (mean3).
IVS = ["mean1", "mean2", "mean3"]

# ---- Panel B: 2SLS (the R ivreg converted to linearmodels.IV2SLS) -------------------
def run_2sls(df, y, endog="ESGR", ivs=IVS, controls=CONTROLS):
    sub = df.dropna(subset=[y, endog] + ivs + controls).copy()
    exog = sm.add_constant(sub[controls])
    res = IV2SLS(sub[y], exog, sub[[endog]], sub[ivs]).fit(cov_type="robust")
    return res, sub

print("\n========== Table 10 Panel B: 2SLS ==========")
res_2sls = {}
for ie in ["IE1", "IE2", "IE3"]:
    r, sub = run_2sls(Statement, ie)
    res_2sls[ie] = r
    # report Panel B shows an Intercept row, so print it alongside ESG_2SLS
    print(f"\n--- {ie} (second stage) ---")
    print(f"  Intercept   = {r.params['const']:.4f} "
          f"(t={r.tstats['const']:.2f}, p={r.pvalues['const']:.4f})")
    print(f"  ESG_2SLS    = {r.params['ESGR']:.4f} "
          f"(t={r.tstats['ESGR']:.2f}, p={r.pvalues['ESGR']:.4f})")

# ---- Panel A: endogeneity / overidentification / weak-IV diagnostics (using IE1) ----
print("\n========== Table 10 Panel A: diagnostic tests (IE1) ==========")
r1 = res_2sls["IE1"]
try:
    wh = r1.wu_hausman()        # Wu-Hausman / DWH endogeneity test
    print(f"Durbin-Wu-Hausman (endogeneity):  F = {wh.stat:.3f}, p = {wh.pval:.4f}")
except Exception as e:
    print("Wu-Hausman:", e)
try:
    print(f"Durbin score:                     {r1.durbin().stat:.3f}, p = {r1.durbin().pval:.4f}")
except Exception:
    pass
try:
    print(f"Sargan (overidentification):      {r1.sargan.stat:.3f}, p = {r1.sargan.pval:.4f}")
    print(f"Basmann (overidentification):     {r1.basmann.stat:.3f}, p = {r1.basmann.pval:.4f}")
except Exception as e:
    print("overid:", e)
# Weak instruments: first-stage F (under robust covariance, approximates the
# Kleibergen-Paap rk Wald F).
fs = r1.first_stage.diagnostics
print("First-stage weak-instrument test (partial R-squared / F):")
print(fs.round(4))

# ---- Panel C: two-step system GMM (L.IE + AR(1)/AR(2) + Hansen) ---------------------
from pydynpd import regression as dynpd

def run_sys_gmm(df, y, esg="ESGR", controls=CONTROLS):
    g = df[["Symbol", "Year", y, esg] + controls].dropna().copy()
    g["id"] = g["Symbol"].astype("category").cat.codes      # pydynpd needs a numeric panel id
    g = g.sort_values(["id", "Year"])
    ctrl = " ".join(controls)
    cmd = (f"{y} L1.{y} {esg} {ctrl} | "
           f"gmm({y}, 2:4) gmm({esg}, 2:4) iv({ctrl}) | timedumm collapse")
    return dynpd.abond(cmd, g, ["id", "Year"])

print("\n========== Table 10 Panel C: two-step system GMM ==========")
for ie in ["IE1", "IE2", "IE3"]:
    print(f"\n----- {ie} -----")
    try:
        run_sys_gmm(Statement, ie)   # prints the coefficient table + AR(1)/AR(2)/Hansen
    except Exception as e:
        print(f"[{ie}] GMM failed: {e} (check the panel is long enough and numpy<2)")


# %% [18.1] Additional test: corporate life cycle -- Table 11 -------------------------
# three subsample regressions. Report Table 11 uses FE + TRANS and splits Panels by
# IE1/IE2/IE3; here we use the FE version throughout.
# NOTE: "企业生命周期" is the life-cycle column in CLC.xlsx; the category labels
#       "成长期/成熟期/衰退期" mean growth/mature/decline and must match the file.
try:
    CLC = pd.read_excel("CLC.xlsx")
    CLC["Symbol"] = pad_symbol(CLC["Symbol"])
    CLC["Year"] = pd.to_numeric(CLC["Year"], errors="coerce").astype("Int64").astype(int)
    CLC = yrfilter(CLC)[["Symbol", "Year", "企业生命周期"]]
    CLC["stage"] = CLC["企业生命周期"].map({"成长期": 3, "成熟期": 2, "衰退期": 1})

    St_clc = lj(Statement, CLC[["Symbol", "Year", "stage"]]).dropna(subset=["stage"])
    stage_name = {3: "growth", 2: "mature", 1: "decline"}
    print("\n========== Table 11: corporate life cycle ==========")
    for ie in ["IE1", "IE2", "IE3"]:
        for s in (3, 2, 1):
            sub = St_clc[St_clc["stage"] == s]
            m = fe_ols(sub, ie, ["ESGR", "TRANS"], CONTROLS)
            print(tidy(m, label=f"{ie} | {stage_name[s]} (N={len(sub)})")
                  .loc[["Intercept", "ESGR", "TRANS"]])
except FileNotFoundError:
    print("\n[Skipped Table 11] CLC.xlsx not found.")


# %% [18.2] Additional test: overinvestment vs underinvestment -- Table 12 ------------
# Over_or_Under_Investment: 1 = overinvestment, 0 = underinvestment (residual sign, Richardson 2006)
print("\n========== Table 12: over / under investment ==========")
over = Statement[Statement["Over_or_Under_Investment"] == 1]
under = Statement[Statement["Over_or_Under_Investment"] == 0]
for ie in ["IE1", "IE2", "IE3"]:
    print(tidy(fe_ols(over,  ie, ["ESGR", "TRANS"], CONTROLS),
               label=f"Overinvestment | {ie} (N={len(over)})").loc[["Intercept", "ESGR", "TRANS"]])
    print(tidy(fe_ols(under, ie, ["ESGR", "TRANS"], CONTROLS),
               label=f"Underinvestment | {ie} (N={len(under)})").loc[["Intercept", "ESGR", "TRANS"]])


# %% [18.3] Additional test: property-rights heterogeneity SOE / non-SOE -- Table 13 --
print("\n========== Table 13: property-rights heterogeneity ==========")
soe = Statement[Statement["SOE"] == 1]
nonsoe = Statement[Statement["SOE"] == 0]
for ie in ["IE1", "IE2", "IE3"]:
    print(tidy(fe_ols(soe,    ie, ["ESGR", "TRANS"], CONTROLS),
               label=f"SOEs | {ie} (N={len(soe)})").loc[["Intercept", "ESGR", "TRANS"]])
    print(tidy(fe_ols(nonsoe, ie, ["ESGR", "TRANS"], CONTROLS),
               label=f"non-SOEs | {ie} (N={len(nonsoe)})").loc[["Intercept", "ESGR", "TRANS"]])


# %% [18.4] ===== [STATA->PY ADDED] Industry heterogeneity -- Table 14 ================
# Split by the industry-code lists given in the report notes. Requires a 'Industry'
# column with the letter+digits industry code. If your Industry has another format,
# adjust industry_letter() accordingly.
HIGH_POLLUTION = {"B06","B07","B08","B09","C17","C19","C22","C25","C26","C28",
                  "C29","C30","C31","C32","D44"}
HIGH_TECH = {"C25","C26","C27","C28","C29","C31","C32","C34","C35","C36","C37",
             "C38","C39","C40","C41","I63","I64","I65","M73"}
LABOR_INT = {"A01","A02","A03","A05","B06","B08","B09","C13","C14","C15","C17",
             "C18","C19","C20","C21","C23","C24","C32","C34","D46","E48","E49",
             "E50","F51","F52","G53","G54","G58","G59","I63","I64","K70","L72",
             "M73","M75","N78","P82","R85","R87","S90"}
TECH_INT = {"N77","C36","M74","I65","C33","C35","C27","C29","C39","C38","C37",
            "C41","C40"}
CAP_INT = {"G56","D44","A04","B11","D45","B07","C22","C31","G55","C30","R86",
           "C28","C26","C25"}

def industry_letter(x):
    """Normalize the industry code to 'letter + 2 digits' (e.g. C25). Adjust as needed."""
    s = str(x).strip().upper()
    return s[:3] if len(s) >= 3 else s

print("\n========== Table 14 Panel A & B: industry heterogeneity ==========")
if "Industry" in Statement.columns:
    St = Statement.copy()
    St["icode"] = St["Industry"].map(industry_letter)
    groups = {
        "High pollution":      St[St["icode"].isin(HIGH_POLLUTION)],
        "Low pollution":       St[~St["icode"].isin(HIGH_POLLUTION)],
        "High-tech":           St[St["icode"].isin(HIGH_TECH)],
        "Traditional":         St[~St["icode"].isin(HIGH_TECH)],
        "Labor-intensive":     St[St["icode"].isin(LABOR_INT)],
        "Technology-intensive":St[St["icode"].isin(TECH_INT)],
        "Capital-intensive":   St[St["icode"].isin(CAP_INT)],
    }
    for name, sub in groups.items():
        if len(sub) > 50:
            for ie in ["IE1", "IE2", "IE3"]:
                m = fe_ols(sub, ie, ["ESGR", "TRANS"], CONTROLS)
                print(tidy(m, label=f"{name} | {ie} (N={len(sub)})").loc[["Intercept", "ESGR", "TRANS"]])
else:
    print("[Skipped Table 14] missing the 'Industry' industry-code column.")


# %% [18.5] Additional test: Confucianism -- Table 15 ---------------------------------
# Use the ready-made confu1..confu7 columns in 儒家文化.xlsx (i.e. confuN from the
# report's Eq.13, the normalized distance index to the nearest N Confucian centers).
# Per the report: take the mean of confu1..confu7 as confu_mean; firms above the sample
# median = high-Confucian group (1), below = low group (0).
# NOTE: "儒家文化.xlsx" is the actual file name and is kept as-is.
print("\n========== Table 15: Confucianism ==========")
try:
    CONFU = pd.read_excel("儒家文化.xlsx")
    CONFU["Symbol"] = pad_symbol(CONFU["Symbol"])
    CONFU["Year"] = pd.to_numeric(CONFU["Year"], errors="coerce").astype("Int64").astype(int)
    CONFU = yrfilter(CONFU)
    confu_cols = [f"confu{i}" for i in range(1, 8)]
    CONFU["confu_mean"] = CONFU[confu_cols].mean(axis=1)

    St_cf = lj(Statement, CONFU[["Symbol", "Year", "confu_mean"]]).dropna(subset=["confu_mean"])
    med = St_cf["confu_mean"].median()
    St_cf["confu_high"] = (St_cf["confu_mean"] >= med).astype(int)

    for grp, lbl in [(1, "High Confucian"), (0, "Low Confucian")]:
        sub = St_cf[St_cf["confu_high"] == grp]
        for ie in ["IE1", "IE2", "IE3"]:
            m = fe_ols(sub, ie, ["ESGR", "TRANS"], CONTROLS)
            print(tidy(m, label=f"{lbl} | {ie} (N={len(sub)})").loc[["Intercept", "ESGR", "TRANS"]])
except FileNotFoundError:
    print("[Skipped Table 15] 儒家文化.xlsx not found.")

print("\n===== All analyses complete =====")

Note: you may need to restart the kernel to use updated packages.
Information-asymmetry PCA -- variance explained (report Table 1a: PC1=46.08%, PC1+PC2=78.87%):
     Eigenvalue  Prop. variance %  Cumulative %
PC1         2.2             73.35         73.35
PC2         0.6             19.98         93.32
PC3         0.2              6.68        100.00
  -> reproduced: PC1=73.35%, PC1+PC2=93.32%

Component loadings (compare with Table 1a):
        PC1     PC2     PC3
LR   0.5761 -0.5836 -0.5723
ILL  0.6297 -0.1296  0.7660
GAM  0.5212  0.8016 -0.2928

N = 19103
           Mean     SD    Min  1stQu  Median  3rdQu    Max
IE1       -0.04   0.05  -0.27  -0.05   -0.02  -0.01  -0.00
IE2       -0.04   0.04  -0.22  -0.05   -0.03  -0.01  -0.00
IE3       -0.04   0.04  -0.22  -0.05   -0.03  -0.01  -0.00
ESGR       4.20   1.01   1.00   4.00    4.00   5.00   6.00
TRANS     -0.00   1.48 -18.54  -0.80    0.05   0.87   7.54
Size      22.46   1.33  20.13  21.50   22.25  23.20  26.67
PB         3.32   2.75

In [ ]:
import sys, numpy
from linearmodels.iv import IV2SLS
import pydynpd
print(sys.executable)
print("numpy", numpy.__version__)

/opt/anaconda3/envs/fyp/bin/python
numpy 1.26.4


In [16]:
# ===== Table 8 (ESGR-only lag specification): Lagged disclosure only, with transparency measured contemporaneously =====
# Mechanism: ESG disclosure in year t−1 affects information transparency in year t; 
# the mediator (TRANS_t) is measured consistently across both paths, allowing for a cleaner mediation decomposition.
#   Model 1   : TRANS_t ~ L1.ESGR + controls + FE              (Path a: lagged disclosure → contemporaneous transparency)
#   Models 2–4: IE_t    ~ L1.ESGR + TRANS_t + controls + FE    (Path b uses contemporaneous TRANS; direct effect is lagged)
import numpy as np
St = Statement.sort_values(["Symbol", "Year"]).copy()
St["Year_prev"] = St.groupby("Symbol")["Year"].shift(1)
consecutive = (St["Year"] - St["Year_prev"] == 1)
St["ESGR_L1"] = St.groupby("Symbol")["ESGR"].shift(1).where(consecutive)   
print(f"Lagged sample N = {St.dropna(subset=['ESGR_L1']).shape[0]} "
      f"(full = {len(Statement)}; drop first-year observations for each firm + observations with discontinuous year gaps)\n")
# Model 1: TRANS_t ~ L1.ESGR
print(tidy(fe_ols(St, "TRANS", ["ESGR_L1"], CONTROLS),
           keep=("ESGR_L1",), label="Table 8 (ESGR lag) Model 1: TRANS ~ L1.ESGR"))
# Models 2-4: IE_t ~ L1.ESGR + TRANS_t   (lag ESGR only)
for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    m = fe_ols(St, ie, ["ESGR_L1", "TRANS"], CONTROLS)
    print(tidy(m, keep=("ESGR_L1", "TRANS"),
               label=f"Table 8 (ESGR lag) {mlbl}: {ie} ~ L1.ESGR + TRANS"))

Lagged sample N = 15064 (full = 19103; drop first-year observations for each firm + observations with discontinuous year gaps)


=== Table 8 (ESGR lag) Model 1: TRANS ~ L1.ESGR | N=15064 | adj_R2=0.6455 ===
              coef        t       p  sig
var                                     
Intercept -19.7521 -51.3297  0.0000  ***
ESGR_L1     0.0210   2.2178  0.0266   **
Size        0.9446  61.1872  0.0000  ***
PB          0.1817  28.6089  0.0000  ***
Lev        -1.6586 -17.5905  0.0000  ***
ROA         1.6865   9.9742  0.0000  ***
Zsorcing   -0.0031  -0.9988  0.3179     
LA          0.0526   2.1482  0.0317   **
Board      -0.0052  -0.6020  0.5472     
IDR         0.0005   0.2062  0.8367     
Age         0.0012   0.2985  0.7653     
Female      0.0002   0.1839  0.8541     
FB         -0.0040  -0.1857  0.8527     
OB          0.0180   0.8113  0.4172     
TOP        -0.0205 -23.4955  0.0000  ***
SECOND     -0.0358 -21.0141  0.0000  ***
DUAL        0.0114   0.5080  0.6115     
SOE        -0.

In [11]:
# ===== Table 8 (lagged specification): one-year lag of ESG disclosure and transparency =====
# The mediation structure follows Table 8, based on the Baron & Kenny framework,
# but the key explanatory variables are lagged by one period:
#   Model 1   : TRANS_t ~ L1.ESGR + controls + FE
#               (path a: lagged ESG disclosure -> current transparency)
#   Model 2-4 : IE_t    ~ L1.ESGR + L1.TRANS + controls + FE
#               (path b + direct effect: lagged mediation path and lagged direct effect)
# Motivation: the transmission mechanism from ESG disclosure to transparency and then to
# investment efficiency may involve a time lag; using lagged explanatory variables also helps
# mitigate potential reverse causality.
# Control variables are kept contemporaneous, reflecting current firm characteristics.
# If any additional variable needs to be lagged, it can be shifted in the same manner.
import numpy as np

St = Statement.sort_values(["Symbol", "Year"]).copy()
St["Year_prev"] = St.groupby("Symbol")["Year"].shift(1)
consecutive = (St["Year"] - St["Year_prev"] == 1)              
for v in ["ESGR", "TRANS"]:
    St[f"{v}_L1"] = St.groupby("Symbol")[v].shift(1).where(consecutive)

print(f"Lagged sample N = {St.dropna(subset=['ESGR_L1']).shape[0]} "
      f"(full = {len(Statement)}; drop first-year observations for each firm + observations with discontinuous year gaps)\n")

# Model 1: TRANS ~ L1.ESGR 
print(tidy(fe_ols(St, "TRANS", ["ESGR_L1"], CONTROLS),
           keep=("ESGR_L1",), label="Table 8 (lag) Model 1: TRANS ~ L1.ESGR"))

# Models 2-4: IE ~ L1.ESGR + L1.TRANS
for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    m = fe_ols(St, ie, ["ESGR_L1", "TRANS_L1"], CONTROLS)
    print(tidy(m, keep=("ESGR_L1", "TRANS_L1"),
               label=f"Table 8 (lag) {mlbl}: {ie} ~ L1.ESGR + L1.TRANS"))

Lagged sample N = 15064 (full = 19103; 每家公司丢首年 + 年份断档行)


=== Table 8 (lag) Model 1: TRANS ~ L1.ESGR | N=15064 | adj_R2=0.6455 ===
              coef        t       p  sig
var                                     
Intercept -19.7521 -51.3297  0.0000  ***
ESGR_L1     0.0210   2.2178  0.0266   **
Size        0.9446  61.1872  0.0000  ***
PB          0.1817  28.6089  0.0000  ***
Lev        -1.6586 -17.5905  0.0000  ***
ROA         1.6865   9.9742  0.0000  ***
Zsorcing   -0.0031  -0.9988  0.3179     
LA          0.0526   2.1482  0.0317   **
Board      -0.0052  -0.6020  0.5472     
IDR         0.0005   0.2062  0.8367     
Age         0.0012   0.2985  0.7653     
Female      0.0002   0.1839  0.8541     
FB         -0.0040  -0.1857  0.8527     
OB          0.0180   0.8113  0.4172     
TOP        -0.0205 -23.4955  0.0000  ***
SECOND     -0.0358 -21.0141  0.0000  ***
DUAL        0.0114   0.5080  0.6115     
SOE        -0.0088  -0.2302  0.8179     

=== Table 8 (lag) Model 2: IE1 ~ L1.ESGR + L1.TR

In [17]:
# ===== Table 8 (two-year lag specification): two-year lag of ESG disclosure and transparency =====
# The model structure is the same as the one-year lag specification above,
# but the key explanatory variables are replaced with two-period lags (L2):
#   Model 1   : TRANS_t ~ L2.ESGR + controls + FE
#               (path a: two-year lagged ESG disclosure -> current transparency)
#   Model 2-4 : IE_t    ~ L2.ESGR + L2.TRANS + controls + FE
#               (path b + direct effect: two-year lagged mediation path and direct effect)
# Motivation: this specification examines the possibility of a longer transmission lag
# and further mitigates potential reverse causality; the trade-off is a further reduction
# in sample size, as one additional year is dropped for each firm.
St2 = Statement.sort_values(["Symbol", "Year"]).copy()
St2["Year_prev2"] = St2.groupby("Symbol")["Year"].shift(2)

consecutive2 = (St2["Year"] - St2["Year_prev2"] == 2)
for v in ["ESGR", "TRANS"]:
    St2[f"{v}_L2"] = St2.groupby("Symbol")[v].shift(2).where(consecutive2)
print(f"Two-year-lag sample N = {St2.dropna(subset=['ESGR_L2']).shape[0]} "
      f"(full = {len(Statement)}; drop the first two years for each firm + observations with discontinuous year gaps)\n")
# Model 1: TRANS ~ L2.ESGR
print(tidy(fe_ols(St2, "TRANS", ["ESGR_L2"], CONTROLS),
           keep=("ESGR_L2",), label="Table 8 (lag2) Model 1: TRANS ~ L2.ESGR"))
# Models 2-4: IE ~ L2.ESGR + L2.TRANS
for ie, mlbl in [("IE1", "Model 2"), ("IE2", "Model 3"), ("IE3", "Model 4")]:
    m = fe_ols(St2, ie, ["ESGR_L2", "TRANS_L2"], CONTROLS)
    print(tidy(m, keep=("ESGR_L2", "TRANS_L2"),
               label=f"Table 8 (lag2) {mlbl}: {ie} ~ L2.ESGR + L2.TRANS"))

Two-year-lag sample N = 11943 (full = 19103; drop the first two years for each firm + observations with discontinuous year gaps)


=== Table 8 (lag2) Model 1: TRANS ~ L2.ESGR | N=11943 | adj_R2=0.6566 ===
              coef        t       p  sig
var                                     
Intercept -19.7526 -45.7364  0.0000  ***
ESGR_L2     0.0104   1.0068  0.3140     
Size        0.9697  57.3471  0.0000  ***
PB          0.1997  27.3898  0.0000  ***
Lev        -1.7438 -16.5362  0.0000  ***
ROA         1.6703   9.1471  0.0000  ***
Zsorcing   -0.0041  -1.1414  0.2537     
LA          0.0402   1.3513  0.1766     
Board      -0.0052  -0.5352  0.5925     
IDR         0.0003   0.1061  0.9155     
Age         0.0012   0.2829  0.7773     
Female     -0.0002  -0.2003  0.8412     
FB         -0.0072  -0.2977  0.7660     
OB          0.0143   0.5792  0.5624     
TOP        -0.0211 -21.7646  0.0000  ***
SECOND     -0.0367 -18.9292  0.0000  ***
DUAL        0.0251   1.0008  0.3169     
SOE        -0.01

In [9]:
%pip install scikit-learn

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from statsmodels.multivariate.pca import PCA as smPCA
import numpy as np

AGENCY_COLS = ["华证", "Wind", "富时罗素", "盟浪", "商道融绿", "msci"]  
VAR_THRESHOLD = 0.70    

_ag = ESG341[["Symbol", "Year"] + AGENCY_COLS].drop_duplicates(["Symbol", "Year"])
St9 = Statement.merge(_ag, on=["Symbol", "Year"], how="left")
St9 = St9[St9[AGENCY_COLS].notna().any(axis=1)].copy()

imp = IterativeImputer(max_iter=50, random_state=0)
filled = pd.DataFrame(imp.fit_transform(St9[AGENCY_COLS]),
                      columns=AGENCY_COLS, index=St9.index)

pc = smPCA(filled, ncomp=len(AGENCY_COLS), standardize=True, method="eig")
prop = np.asarray(pc.eigenvals).ravel(); prop = prop / prop.sum()
cum = np.cumsum(prop)

k = int(np.argmax(cum >= VAR_THRESHOLD)) + 1 if (cum >= VAR_THRESHOLD).any() else len(prop)
w = prop[:k] / prop[:k].sum()                       
esgid = pc.factors.iloc[:, :k].values @ w          

if np.corrcoef(esgid, St9["ESGR"])[0, 1] < 0:
    esgid = -esgid
St9["ESGID"] = winsorize(pd.Series(esgid, index=St9.index))

print(pd.DataFrame({"Prop.%": prop*100, "Cum.%": cum*100},
                   index=[f"PC{i+1}" for i in range(len(AGENCY_COLS))]).round(2))
print(f"Components retained (cum >= {VAR_THRESHOLD:.0%}): k = {k} | weights = {np.round(w,3)}")
print(f"ESGID sample N = {len(St9)} | corr(ESGID, ESGR) = {np.corrcoef(St9['ESGID'], St9['ESGR'])[0,1]:.3f}")

print(tidy(fe_ols(St9, "TRANS", ["ESGID"], CONTROLS),
           label="Table 9 Model1: TRANS ~ ESGID").loc[["Intercept", "ESGID"]])
for ie, mlbl in [("IE1", "Model2"), ("IE2", "Model3"), ("IE3", "Model4")]:
    m = fe_ols(St9, ie, ["ESGID", "TRANS"], CONTROLS)
    print(tidy(m, label=f"Table 9 {mlbl}: {ie} ~ ESGID + TRANS").loc[["Intercept", "ESGID", "TRANS"]])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 13.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
     Prop.%   Cum.%
PC1   76.62   76.62
PC2   11.67   88.29
PC3    4.77   93.06
PC4    4.31   97.38
PC5    2.11   99.49
PC6    0.51  100.00
Components retained (cum >= 70%): k = 1 | weights = [1.]
ESGID sample N = 19103 | corr(ESGID, ESGR) = 0.654

=== Table 9 Model1: TRANS ~ ESGID | N=19103 | adj_R2=0.6275 ===
              coef        t    p  sig
var                                  
Intercept -20.2356 -56.4418  0.0  ***
ESGID       6.6426   4.4983  0.0  ***

=== Table 9 Model2: IE1 ~ ESGID + TRANS | N=19103 | adj_R2=0.1394 ===
             coef       t       p  sig
var                                   
Intercept -0.0822 -5.2314  0.0000  ***
ESGID      0.2630  4.9785  0.0000  ***
TRANS      0.0004  1.0613  0.2885     

=== Table 9 Model3: IE2 ~ ESGID + TRAN

In [5]:
# ---- Panel B first stage: ESGR ~ ESG_1 + ESG_2 + ESG_3 + controls (+FE) ----
# (identical across IE1/IE2/IE3 since the dependent variable here is ESGR, so report once)
fs1 = res_2sls["IE1"].first_stage.individual["ESGR"]
fs_rows = ["const"] + IVS              
fs_tbl = pd.DataFrame({"coef": fs1.params, "t": fs1.tstats, "p": fs1.pvalues}).loc[fs_rows]
fs_tbl["sig"] = pd.cut(fs_tbl["p"], [-1, .01, .05, .1, 1], labels=["***", "**", "*", ""])
fs_tbl.index = ["Intercept", "ESG_1 (mean1)", "ESG_2 (mean2)", "ESG_3 (mean3)"]
print("\n--- Table 10 Panel B first stage (dep. var = ESGR) ---")
print(fs_tbl.round(4))
print(f"first-stage adj. R^2 = {fs1.rsquared_adj:.4f}, N = {int(fs1.nobs)}")


--- Table 10 Panel B first stage (dep. var = ESGR) ---
                 coef        t      p  sig
Intercept     -5.7070 -27.3444  0.000  ***
ESG_1 (mean1)  0.1758   6.0842  0.000  ***
ESG_2 (mean2) -0.0848  -2.5135  0.012   **
ESG_3 (mean3)  0.9135  45.4186  0.000  ***
first-stage adj. R^2 = 0.3237, N = 19103
